# Transparent Conductors Predictions
> **John El Berch**



#### Why this project?
This application aligns well with my Ph.D.'s research topic (computational catalysis) and, beyond the learning experience, I believe it would be great to feature this project as part of my digital portfolio. 


#### Experience
My experience with the topic comes from my Ph.D. research, focused on the discovery of novel catalysts through computational chemistry. Although this application (focused on semiconductor discovery) doesn't exactly match my research area, many concepts are shared (specially, those related to material properties). As a newcomer in the data science field (starting in CMPINF 2100 last semester), I haven't worked on a data-based project for an application similar to this.


#### Goals
Predict two target properties (Formation energy and Bandgap energy) for promising transparent conductors based on material properties such as the spacegroup, total number of atoms, relative compositions...


#### References and resources
Project based on [Kaggle](https://www.kaggle.com/competitions/nomad2018-predict-transparent-conductors/rules) competition. For the project, two data sets were provided, a `train.csv` and a hold-out `test.csv` dataset. Both are available in the provided link upon agreeing to the competitions' rules. Both datasets will be explored, while the `train.csv` will be used to test for different models' effectiveness. A separate `submission.csv` file will be available at the main GitHub, which will contain all the predicted values as:

```
id,formation_energy_ev_natom,bandgap_energy_ev
1,0.1779,1.8892
2,0.1779,1.8892
3,0.1779,1.8892
...
```

Finally, a `score.txt` file will be submitted with the score obtained from Kaggle.

***
# Exploratory Data Analysis (EDA)
### Preliminars

This notebook will explore basic properties of the two provided datasets

### Import modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Read the training and testing data sets
##### `train.csv`

In [2]:
df_train = pd.read_csv('../train.csv')
df_train.head()

,id,spacegroup,number_of_total_atoms,percent_atom_al,percent_atom_ga,percent_atom_in,lattice_vector_1_ang,lattice_vector_2_ang,lattice_vector_3_ang,lattice_angle_alpha_degree,lattice_angle_beta_degree,lattice_angle_gamma_degree,formation_energy_ev_natom,bandgap_energy_ev
0,1,33,80.0,0.6250,0.3750,0.000,9.9523,8.5513,9.1775,90.0026,90.0023,90.0017,0.0680,3.4387
1,2,194,80.0,0.6250,0.3750,0.000,6.1840,6.1838,23.6287,90.0186,89.9980,120.0025,0.2490,2.9210
2,3,227,40.0,0.8125,0.1875,0.000,9.7510,5.6595,13.9630,90.9688,91.1228,30.5185,0.1821,2.7438
3,4,167,30.0,0.7500,0.0000,0.250,5.0036,5.0034,13.5318,89.9888,90.0119,120.0017,0.2172,3.3492
4,5,194,80.0,0.0000,0.6250,0.375,6.6614,6.6612,24.5813,89.9960,90.0006,119.9893,0.0505,1.3793


##### `test.csv`

In [3]:
df_test = pd.read_csv('../test.csv')
df_test.head()

,id,spacegroup,number_of_total_atoms,percent_atom_al,percent_atom_ga,percent_atom_in,lattice_vector_1_ang,lattice_vector_2_ang,lattice_vector_3_ang,lattice_angle_alpha_degree,lattice_angle_beta_degree,lattice_angle_gamma_degree
0,1,33,80.0,0.1875,0.4688,0.3438,10.5381,9.0141,9.6361,89.9997,90.0003,90.0006
1,2,33,80.0,0.7500,0.2500,0.0000,9.8938,8.5014,9.1298,90.0038,90.0023,90.0015
2,3,167,30.0,0.6667,0.1667,0.1667,4.9811,4.9808,13.4799,89.9900,90.0109,120.0014
3,4,12,80.0,0.5625,0.4375,0.0000,24.3370,6.0091,5.7620,89.9995,103.8581,90.0002
4,5,12,80.0,0.1875,0.5000,0.3125,24.6443,6.2906,6.1589,90.0000,104.5929,90.0001


→ Treat the `spacegroup` variable as a categorical. Create a categorical version of the `number_of_total_atoms` and also create an additional categorical variable splitting the `number_of_atoms` variable as 80 or other.

In [4]:
df_train['spacegroup'] = df_train['spacegroup'].astype('category')
df_train['number_of_total_atoms_cat'] = df_train['number_of_total_atoms'].astype('category')
df_train['binary_number_of_total_atoms'] = np.where(df_train.number_of_total_atoms.to_numpy() == 80, '80', 'other')

In [5]:
df_test['spacegroup'] = df_test['spacegroup'].astype('category')
df_test['number_of_total_atoms_cat'] = df_test['number_of_total_atoms'].astype('category')
df_test['binary_number_of_total_atoms'] = np.where(df_test.number_of_total_atoms.to_numpy() == 80, '80', 'other')

***
## Preliminars

#### Number of rows and columns in the data sets
`train.csv`

In [6]:
print(f'Number of rows = {df_train.shape[0]}\nNumber of columns = {df_train.shape[1]}')

Number of rows = 2400
Number of columns = 16


`test.csv`

In [7]:
print(f'Number of rows = {df_test.shape[0]}\nNumber of columns = {df_test.shape[1]}')

Number of rows = 600
Number of columns = 14


#### Variable names

`train.csv`

In [8]:
df_train.columns.to_list()

['id',
 'spacegroup',
 'number_of_total_atoms',
 'percent_atom_al',
 'percent_atom_ga',
 'percent_atom_in',
 'lattice_vector_1_ang',
 'lattice_vector_2_ang',
 'lattice_vector_3_ang',
 'lattice_angle_alpha_degree',
 'lattice_angle_beta_degree',
 'lattice_angle_gamma_degree',
 'formation_energy_ev_natom',
 'bandgap_energy_ev',
 'number_of_total_atoms_cat',
 'binary_number_of_total_atoms']

`test.csv`

In [9]:
df_test.columns.to_list()

['id',
 'spacegroup',
 'number_of_total_atoms',
 'percent_atom_al',
 'percent_atom_ga',
 'percent_atom_in',
 'lattice_vector_1_ang',
 'lattice_vector_2_ang',
 'lattice_vector_3_ang',
 'lattice_angle_alpha_degree',
 'lattice_angle_beta_degree',
 'lattice_angle_gamma_degree',
 'number_of_total_atoms_cat',
 'binary_number_of_total_atoms']

Differences between the two datasets

In [10]:
[element for element in df_train.columns.to_list() if element not in df_test.columns.to_list()]

['formation_energy_ev_natom', 'bandgap_energy_ev']

The difference is that `test.csv` lacks the output variables

#### Identify which variables are inputs, outputs, and which variables are associated with unique identifying information

* Outputs
    - `formation_energy_ev_natom`
    - `bandgap_energy_ev`
<br>
* Inputs: All other variables (except `id`) are inputs
* Variables associated with unique identifying information: `spacegroup` (input that specifies what is the configuration of the atoms in space)

#### Data types of the variables

`train.csv`

In [11]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   id                            2400 non-null   int64   
 1   spacegroup                    2400 non-null   category
 2   number_of_total_atoms         2400 non-null   float64 
 3   percent_atom_al               2400 non-null   float64 
 4   percent_atom_ga               2400 non-null   float64 
 5   percent_atom_in               2400 non-null   float64 
 6   lattice_vector_1_ang          2400 non-null   float64 
 7   lattice_vector_2_ang          2400 non-null   float64 
 8   lattice_vector_3_ang          2400 non-null   float64 
 9   lattice_angle_alpha_degree    2400 non-null   float64 
 10  lattice_angle_beta_degree     2400 non-null   float64 
 11  lattice_angle_gamma_degree    2400 non-null   float64 
 12  formation_energy_ev_natom     2400 non-null   fl

`test.csv`

In [12]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   id                            600 non-null    int64   
 1   spacegroup                    600 non-null    category
 2   number_of_total_atoms         600 non-null    float64 
 3   percent_atom_al               600 non-null    float64 
 4   percent_atom_ga               600 non-null    float64 
 5   percent_atom_in               600 non-null    float64 
 6   lattice_vector_1_ang          600 non-null    float64 
 7   lattice_vector_2_ang          600 non-null    float64 
 8   lattice_vector_3_ang          600 non-null    float64 
 9   lattice_angle_alpha_degree    600 non-null    float64 
 10  lattice_angle_beta_degree     600 non-null    float64 
 11  lattice_angle_gamma_degree    600 non-null    float64 
 12  number_of_total_atoms_cat     600 non-null    cate

#### Number of missing values per variable

`train.csv`

In [13]:
df_train.isna().sum()

id                              0
spacegroup                      0
number_of_total_atoms           0
percent_atom_al                 0
percent_atom_ga                 0
percent_atom_in                 0
lattice_vector_1_ang            0
lattice_vector_2_ang            0
lattice_vector_3_ang            0
lattice_angle_alpha_degree      0
lattice_angle_beta_degree       0
lattice_angle_gamma_degree      0
formation_energy_ev_natom       0
bandgap_energy_ev               0
number_of_total_atoms_cat       0
binary_number_of_total_atoms    0
dtype: int64

`test.csv`

In [14]:
df_test.isna().sum()

id                              0
spacegroup                      0
number_of_total_atoms           0
percent_atom_al                 0
percent_atom_ga                 0
percent_atom_in                 0
lattice_vector_1_ang            0
lattice_vector_2_ang            0
lattice_vector_3_ang            0
lattice_angle_alpha_degree      0
lattice_angle_beta_degree       0
lattice_angle_gamma_degree      0
number_of_total_atoms_cat       0
binary_number_of_total_atoms    0
dtype: int64

#### Number of unique values per variable

`train.csv`

In [15]:
df_train.nunique()

id                              2400
spacegroup                         6
number_of_total_atoms              6
percent_atom_al                   42
percent_atom_ga                   42
percent_atom_in                   42
lattice_vector_1_ang            1288
lattice_vector_2_ang            1216
lattice_vector_3_ang            1210
lattice_angle_alpha_degree       457
lattice_angle_beta_degree        566
lattice_angle_gamma_degree       434
formation_energy_ev_natom       1733
bandgap_energy_ev               2307
number_of_total_atoms_cat          6
binary_number_of_total_atoms       2
dtype: int64

`test.csv`

In [16]:
df_test.nunique()

id                              600
spacegroup                        6
number_of_total_atoms             6
percent_atom_al                  42
percent_atom_ga                  40
percent_atom_in                  40
lattice_vector_1_ang            476
lattice_vector_2_ang            462
lattice_vector_3_ang            458
lattice_angle_alpha_degree      231
lattice_angle_beta_degree       267
lattice_angle_gamma_degree      215
number_of_total_atoms_cat         6
binary_number_of_total_atoms      2
dtype: int64